# Program Control Notebook

This notebook is the parent-level operating point for the CE 519 program. All seven modules are now represented here so the full workflow can be executed from one location.

Current active modules:

- Module 1: Steel Reinforced Concrete Pavement Design
- Module 2: Fiber Reinforced Concrete Pavement Design
- Module 3: Life Cycle Costing
- Module 4: Life Cycle Assessment
- Module 5: Uncertainty and Sensitivity Analysis
- Module 6: Optimization and Selection
- Module 7: Summary Output Graphics (Not Yet Implemented)


## Module 1 - Steel Reinforced Concrete Pavement

This cell imports the Module 1 inputs and runner. The default project geometry is already included in `SRCInputs`:

- Parking lot: 300 ft x 150 ft
- Roadway: 2,050 ft x 20 ft
- Total pavement area: 86,000 sf
- Design axle load: 19 kip
- Wheel load: 9.5 kip

Module 1 steel quantities include two-way reinforcement, 40-ft stock rebar lengths, and Class A lap splices. Pavement thickness candidates are limited to even values from 6 to 12 inches.


Reinforcement quantity basis: the parking lot is quantified with two-way reinforcement, while the roadway is quantified with one-way longitudinal reinforcement. Both include 40-ft stock bars and Class A lap splices.


In [1]:
from pathlib import Path
import pandas as pd

from module_1 import SRCInputs, run_module_1_src
from module_1.src_design import write_results_csv


## Module 1 inputs

Edit these values here when running scenarios. Keeping scenario inputs in this parent notebook avoids burying run-critical values inside the module source code or README.


In [2]:
inputs = SRCInputs(
    # Project geometry
    parking_lot_length_ft=300.0,
    parking_lot_width_ft=150.0,
    roadway_length_ft=2050.0,
    roadway_width_ft=20.0,

    # Loading
    axle_load_kip=19.0,
    tire_pressure_psi=100.0,

    # Materials and support
    concrete_fpc_psi=4500.0,
    concrete_MR_psi=650.0,
    subgrade_k_pci=100.0,

    # Reinforcement placement
    clear_cover_bottom_in=3.0,
)

# Module 1 quantities include two-way reinforcement with 40-ft stock bars and Class A lap splices.
reinforcement_directions = 2

print(f'Total CE 519 pavement area = {inputs.total_area_sf:,.0f} sf')
print(f'Design wheel load = {inputs.wheel_load_lb / 1000:.2f} kip')


Total CE 519 pavement area = 86,000 sf
Design wheel load = 9.50 kip


## Run Module 1

This evaluates all discrete Module 1 alternatives and separates feasible results from the full candidate list.


In [3]:
all_results, feasible_results = run_module_1_src(
    inputs,
    reinforcement_directions=reinforcement_directions,
)

print(f'Total candidates checked: {len(all_results):,}')
print(f'Feasible candidates: {len(feasible_results):,}')


Total candidates checked: 560
Feasible candidates: 400


## Save Module 1 outputs

The output CSV files are written to the parent-level `outputs` folder. Later modules can read these files or use the in-memory DataFrames directly.


In [4]:
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)

write_results_csv(all_results, output_dir / 'module_1_all_results.csv')
write_results_csv(feasible_results, output_dir / 'module_1_feasible_results.csv')

all_df = pd.DataFrame(all_results)
feasible_df = pd.DataFrame(feasible_results)

print('Saved:')
print(output_dir / 'module_1_all_results.csv')
print(output_dir / 'module_1_feasible_results.csv')


Saved:
outputs\module_1_all_results.csv
outputs\module_1_feasible_results.csv


## Review feasible Module 1 results

The table below sorts feasible alternatives by demand/capacity ratio and concrete volume. This is only an initial engineering review view; final ranking should occur in the later selection/optimization module.


In [5]:
review_columns = [
    'subbase_thickness_in',
    'pavement_thickness_in',
    'bar_size',
    'bar_spacing_in',
    'demand_capacity_ratio',
    'cracking_ratio',
    'concrete_volume_cy_total',
    'subbase_volume_cy_total',
    'steel_weight_ton_total',
]

feasible_df[review_columns].sort_values(
    by=['pavement_thickness_in', 'steel_weight_ton_total', 'demand_capacity_ratio']
).head(20)


,subbase_thickness_in,pavement_thickness_in,bar_size,bar_spacing_in,demand_capacity_ratio,cracking_ratio,concrete_volume_cy_total,subbase_volume_cy_total,steel_weight_ton_total
321,12,6,#4,8,0.837433,0.740333,1592.592593,3185.185185,71.151431
241,10,6,#4,8,0.845304,0.747291,1592.592593,2654.320988,71.151431
161,8,6,#4,8,0.853758,0.754765,1592.592593,2123.456790,71.151431
81,6,6,#4,8,0.862890,0.762838,1592.592593,1592.592593,71.151431
1,4,6,#4,8,0.872816,0.771613,1592.592593,1061.728395,71.151431
331,12,6,#6,18,0.898930,0.740333,1592.592593,3185.185185,73.483488
251,10,6,#6,18,0.907379,0.747291,1592.592593,2654.320988,73.483488
171,8,6,#6,18,0.916454,0.754765,1592.592593,2123.456790,73.483488
91,6,6,#6,18,0.926256,0.762838,1592.592593,1592.592593,73.483488
11,4,6,#6,18,0.936911,0.771613,1592.592593,1061.728395,73.483488


## Module 2 - Fiber Reinforced Concrete Pavement

Module 2 uses the same project geometry, loading, concrete properties, and support assumptions as Module 1 where appropriate. The FRC decision variable is `fe3`, the equivalent/residual flexural strength input.

Pavement thickness candidates are limited to even values from 6 to 12 inches.


In [6]:
from module_2 import FRCInputs, generate_frc_candidates, run_module_2_frc
from module_2.frc_design import write_results_csv as write_frc_results_csv


## Module 2 inputs

These inputs mirror the appropriate Module 1 inputs. Update `fe3_values_psi` when project-specific FRC performance values are selected.


In [7]:
frc_inputs = FRCInputs(
    # Project geometry
    parking_lot_length_ft=inputs.parking_lot_length_ft,
    parking_lot_width_ft=inputs.parking_lot_width_ft,
    roadway_length_ft=inputs.roadway_length_ft,
    roadway_width_ft=inputs.roadway_width_ft,

    # Loading
    axle_load_kip=inputs.axle_load_kip,
    tire_pressure_psi=inputs.tire_pressure_psi,

    # Materials and support
    concrete_fpc_psi=inputs.concrete_fpc_psi,
    concrete_MR_psi=inputs.concrete_MR_psi,
    concrete_Ec_psi=inputs.concrete_Ec_psi,
    poisson_ratio=inputs.poisson_ratio,
    subgrade_k_pci=inputs.subgrade_k_pci,
    phi_frc=1.0,
)

# Adjust these if project-specific FRC performance values are updated.
fe3_values_psi = [100, 150, 200, 250, 300, 350, 400]

frc_candidates = generate_frc_candidates(
    fe3_values_psi=fe3_values_psi,
)

print(f'Total CE 519 pavement area = {frc_inputs.total_area_sf:,.0f} sf')
print(f'Design wheel load = {frc_inputs.wheel_load_lb / 1000:.2f} kip')
print(f'FRC fe3 values checked = {fe3_values_psi}')


Total CE 519 pavement area = 86,000 sf
Design wheel load = 9.50 kip
FRC fe3 values checked = [100, 150, 200, 250, 300, 350, 400]


## Run Module 2

This evaluates all discrete Module 2 alternatives and separates feasible results from the full candidate list.


In [8]:
module_2_all_results, module_2_feasible_results = run_module_2_frc(
    frc_inputs,
    candidates=frc_candidates,
)

print(f'Total FRC candidates checked: {len(module_2_all_results):,}')
print(f'Feasible FRC candidates: {len(module_2_feasible_results):,}')


Total FRC candidates checked: 140
Feasible FRC candidates: 140


## Save Module 2 outputs

The output CSV files are written to the parent-level `outputs` folder. Later modules can read these files or use the in-memory DataFrames directly.


In [9]:
write_frc_results_csv(module_2_all_results, output_dir / 'module_2_all_results.csv')
write_frc_results_csv(module_2_feasible_results, output_dir / 'module_2_feasible_results.csv')

module_2_all_df = pd.DataFrame(module_2_all_results)
module_2_feasible_df = pd.DataFrame(module_2_feasible_results)

print('Saved:')
print(output_dir / 'module_2_all_results.csv')
print(output_dir / 'module_2_feasible_results.csv')


Saved:
outputs\module_2_all_results.csv
outputs\module_2_feasible_results.csv


## Review feasible Module 2 results

This view sorts feasible FRC alternatives by pavement thickness, fe3, and demand/capacity ratio.


In [10]:
frc_review_columns = [
    'subbase_thickness_in',
    'pavement_thickness_in',
    'fe3_psi',
    'Re3_percent',
    'Mu_kip_in_per_ft',
    'Mn_FRC_kip_in_per_ft',
    'Mtotal_FRC_kip_in_per_ft',
    'phi_Mtotal_FRC_kip_in_per_ft',
    'demand_capacity_ratio',
    'cracking_ratio',
    'concrete_volume_cy_total',
    'subbase_volume_cy_total',
]

module_2_feasible_df[frc_review_columns].sort_values(
    by=['pavement_thickness_in', 'fe3_psi', 'demand_capacity_ratio']
).head(20)


,subbase_thickness_in,pavement_thickness_in,fe3_psi,Re3_percent,Mu_kip_in_per_ft,Mn_FRC_kip_in_per_ft,Mtotal_FRC_kip_in_per_ft,phi_Mtotal_FRC_kip_in_per_ft,demand_capacity_ratio,cracking_ratio,concrete_volume_cy_total,subbase_volume_cy_total
112,12,6,100,15.384615,34.647577,7.2,54.0,54.0,0.641622,0.740333,1592.592593,3185.185185
84,10,6,100,15.384615,34.973218,7.2,54.0,54.0,0.647652,0.747291,1592.592593,2654.320988
56,8,6,100,15.384615,35.323001,7.2,54.0,54.0,0.654130,0.754765,1592.592593,2123.456790
28,6,6,100,15.384615,35.700795,7.2,54.0,54.0,0.661126,0.762838,1592.592593,1592.592593
0,4,6,100,15.384615,36.111482,7.2,54.0,54.0,0.668731,0.771613,1592.592593,1061.728395
113,12,6,150,23.076923,34.647577,10.8,57.6,57.6,0.601520,0.740333,1592.592593,3185.185185
85,10,6,150,23.076923,34.973218,10.8,57.6,57.6,0.607174,0.747291,1592.592593,2654.320988
57,8,6,150,23.076923,35.323001,10.8,57.6,57.6,0.613247,0.754765,1592.592593,2123.456790
29,6,6,150,23.076923,35.700795,10.8,57.6,57.6,0.619805,0.762838,1592.592593,1592.592593
1,4,6,150,23.076923,36.111482,10.8,57.6,57.6,0.626935,0.771613,1592.592593,1061.728395


## Module 3 - Life Cycle Costing

Module 3 consumes the feasible Module 1 and Module 2 alternatives and calculates present-worth life-cycle cost. For this project, maintenance is set to zero, and end-of-life includes full demolition for reuse as crushed concrete aggregate.


In [11]:
from module_3 import LCCInputs, RSMeansUnitCosts, run_module_3_lcc
from module_3.lcc_design import write_results_csv as write_lcc_results_csv


## Module 3 cost inputs

Update these values using the RSMeans line items selected for the CE 519 program. Enter national-average RSMeans values first; the Saginaw County factor is applied separately.


In [12]:
rsmeans_costs = RSMeansUnitCosts(
    # Adjust these baseline values with the selected RSMeans line items.
    concrete_cost_per_cy=185.00,
    stone_57_cost_per_cy=58.00,
    reinforcing_steel_cost_per_ton=3200.00,

    # Baseline FRC fiber cost uses the maximum documented contract value.
    # Module 5 samples $1.323 to $1.47/lb as potential volume-discount savings.
    tufstrand_sf_cost_per_lb=1.47,

    # End-of-service-life costs.
    concrete_demolition_cost_per_cy=42.00,
    concrete_crushing_cost_per_ton=9.00,
    subbase_removal_cost_per_cy=18.00,
    recycled_concrete_aggregate_credit_per_ton=6.00,
)

lcc_inputs = LCCInputs(
    analysis_period_yr=50,
    end_of_service_life_yr=50,
    real_discount_rate=0.03,

    # Enter the current RSMeans / CCI factor for Saginaw County or nearest listed city.
    saginaw_county_location_factor=1.00,

    # No maintenance by project basis.
    maintenance_cost_present_worth=0.0,
)


## Run Module 3

This uses the in-memory feasible results from Modules 1 and 2. If those cells have not been run, run Modules 1 and 2 first.


In [13]:
module_3_lcc_results = run_module_3_lcc(
    module_1_results=feasible_results,
    module_2_results=module_2_feasible_results,
    costs=rsmeans_costs,
    inputs=lcc_inputs,
)

module_3_lcc_df = pd.DataFrame(module_3_lcc_results)

print(f'LCC alternatives evaluated: {len(module_3_lcc_results):,}')


LCC alternatives evaluated: 540


## Save Module 3 outputs


In [14]:
write_lcc_results_csv(module_3_lcc_results, output_dir / 'module_3_lcc_results.csv')

print('Saved:')
print(output_dir / 'module_3_lcc_results.csv')


Saved:
outputs\module_3_lcc_results.csv


## Review lowest present-worth alternatives


In [15]:
lcc_review_columns = [
    'module',
    'subbase_thickness_in',
    'pavement_thickness_in',
    'bar_size',
    'bar_spacing_in',
    'fe3_psi',
    'initial_construction_cost',
    'end_of_life_present_worth',
    'total_present_worth',
    'present_worth_per_sf',
]

available_lcc_columns = [c for c in lcc_review_columns if c in module_3_lcc_df.columns]

module_3_lcc_df[available_lcc_columns].sort_values(
    by=['total_present_worth']
).head(20)


,module,subbase_thickness_in,pavement_thickness_in,bar_size,bar_spacing_in,fe3_psi,initial_construction_cost,end_of_life_present_worth,total_present_worth,present_worth_per_sf
400,FRC,4,6,NaN,NaN,100.0,363233.209877,21824.144859,385057.354735,4.477411
401,FRC,4,6,NaN,NaN,150.0,364169.654321,21824.144859,385993.799180,4.488300
402,FRC,4,6,NaN,NaN,200.0,367681.320988,21824.144859,389505.465847,4.529133
403,FRC,4,6,NaN,NaN,250.0,371192.987654,21824.144859,393017.132513,4.569967
404,FRC,4,6,NaN,NaN,300.0,374704.654321,21824.144859,396528.799180,4.610800
405,FRC,4,6,NaN,NaN,350.0,378216.320988,21824.144859,400040.465847,4.651633
406,FRC,4,6,NaN,NaN,400.0,381727.987654,21824.144859,403552.132513,4.692467
428,FRC,6,6,NaN,NaN,100.0,394023.333333,24003.834732,418027.168066,4.860781
429,FRC,6,6,NaN,NaN,150.0,394959.777778,24003.834732,418963.612510,4.871670
430,FRC,6,6,NaN,NaN,200.0,398471.444444,24003.834732,422475.279177,4.912503


## Module 4 - Life Cycle Assessment

Module 4 calculates total project climate change impact in kg CO2-eq for feasible Module 1 and Module 2 alternatives. The functional unit is one complete CE 519 pavement project.


In [16]:
from module_4 import LCAInputs, LCAUnitImpacts, FacilityLocation, run_module_4_lca
from module_4.lca_design import (
    geocode_address_nominatim,
    route_distance_miles_osrm,
    write_results_csv as write_lca_results_csv,
)


## Module 4 locations and haul distances

Short names are used below for the CE 519 haul-distance inputs. The OpenStreetMap/OSRM cell can be run when internet access is available. If it is not run, the default distances in `LCAInputs` are used.


In [17]:
project_site = FacilityLocation(
    short_name='Project site',
    address='561 N Hemlock Rd, Hemlock, MI 48626',
)

stone_source = FacilityLocation(
    short_name='Wirt Stone Dock',
    address='Wirt Stone Dock, Saginaw, MI',
    latitude=43.4738915,
    longitude=-83.9080425,
)

concrete_source = FacilityLocation(
    short_name='R & R Ready Mix',
    address='R & R Ready Mix Inc, Saginaw, MI',
    latitude=43.4114697,
    longitude=-84.2327798,
)

print(stone_source)
rebar_source = FacilityLocation(
    short_name='HYMMCO',
    address='HYMMCO, Saginaw, MI',
    latitude=43.5037881,
    longitude=-83.9727579,
)

nucor_source = FacilityLocation(
    short_name='Nucor Marion OH',
    address='Nucor Marion, OH',
    latitude=40.5887,
    longitude=-83.1285,
)

print(concrete_source)
print(nucor_source)
print(rebar_source)
print(project_site)


FacilityLocation(short_name='Wirt Stone Dock', address='Wirt Stone Dock, Saginaw, MI', latitude=43.4738915, longitude=-83.9080425)
FacilityLocation(short_name='R & R Ready Mix', address='R & R Ready Mix Inc, Saginaw, MI', latitude=43.4114697, longitude=-84.2327798)
FacilityLocation(short_name='Nucor Marion OH', address='Nucor Marion, OH', latitude=40.5887, longitude=-83.1285)
FacilityLocation(short_name='HYMMCO', address='HYMMCO, Saginaw, MI', latitude=43.5037881, longitude=-83.9727579)
FacilityLocation(short_name='Project site', address='561 N Hemlock Rd, Hemlock, MI 48626', latitude=None, longitude=None)


### Calculate route distances using OpenStreetMap

Run this cell with internet access to calculate one-way route distances. The #57 stone haul distance and crushed concrete haul distance use the same Wirt Stone Dock route.


In [18]:
# Online distance calculation.
# If this fails because internet access is unavailable, manually enter distances in the next cell.

try:
    project_lat, project_lon = geocode_address_nominatim(project_site.address)

    stone_to_project_miles = route_distance_miles_osrm(
        stone_source.latitude,
        stone_source.longitude,
        project_lat,
        project_lon,
    )

    concrete_to_project_miles = route_distance_miles_osrm(
        concrete_source.latitude,
        concrete_source.longitude,
        project_lat,
        project_lon,
    )

    crushed_concrete_to_reuse_miles = stone_to_project_miles
    rebar_to_project_miles = 12.0
    nucor_to_hymmco_miles = 195.0

    rebar_to_project_miles = route_distance_miles_osrm(
        rebar_source.latitude,
        rebar_source.longitude,
        project_lat,
        project_lon,
    )

    nucor_to_hymmco_miles = route_distance_miles_osrm(
        nucor_source.latitude,
        nucor_source.longitude,
        rebar_source.latitude,
        rebar_source.longitude,
    )

    print(f'Project coordinates: {project_lat:.6f}, {project_lon:.6f}')
    print(f'Wirt Stone Dock to project: {stone_to_project_miles:.2f} miles')
    print(f'R & R Ready Mix to project: {concrete_to_project_miles:.2f} miles')
    print(f'Crushed concrete to Wirt Stone Dock: {crushed_concrete_to_reuse_miles:.2f} miles')
    print(f'Nucor Marion OH to HYMMCO: {nucor_to_hymmco_miles:.2f} miles')
    print(f'HYMMCO rebar to project: {rebar_to_project_miles:.2f} miles')

except Exception as exc:
    print('Online distance calculation did not complete.')
    print(exc)
    stone_to_project_miles = 16.0
    concrete_to_project_miles = 18.0
    crushed_concrete_to_reuse_miles = stone_to_project_miles
    rebar_to_project_miles = 12.0
    nucor_to_hymmco_miles = 195.0


Project coordinates: 43.419344, -84.229882
Wirt Stone Dock to project: 19.78 miles
R & R Ready Mix to project: 0.67 miles
Crushed concrete to Wirt Stone Dock: 19.78 miles
Nucor Marion OH to HYMMCO: 236.02 miles
HYMMCO rebar to project: 19.34 miles


## Module 4 LCA inputs

Update the unit impact factors once the final ecoinvent APOS AO / SimaPro factors are selected. TUF-STRAND SF GWP is set from Euclid Chemical's technical data sheet.


In [19]:
lca_unit_impacts = LCAUnitImpacts(
    # Replace ecoinvent factors with final APOS AO/SimaPro results when available.
    concrete_kgco2e_per_cy=355.0,
    stone_57_kgco2e_per_ton=5.0,
    # CRSI fabricated rebar EPD A1-A3: 854 kg CO2-eq/metric ton = 774.736 kg CO2-eq/short ton.
    reinforcing_steel_kgco2e_per_ton=774.736,

    # Euclid TUF-STRAND SF TDS: 3.08 kg CO2-eq/kg.
    tufstrand_sf_kgco2e_per_kg=3.08,

    truck_transport_kgco2e_per_ton_mile=0.17,
    demolition_kgco2e_per_cy_concrete=3.0,
    concrete_crushing_kgco2e_per_ton=1.5,
    virgin_aggregate_credit_kgco2e_per_ton=5.0,
)

# TUF-STRAND dosage is calculated in Module 4 as:
# dosage, lb/yd3 = 0.03*fe3 - 1.1, limited to 3 to 20 lb/yd3.
lca_inputs = LCAInputs(
    stone_to_project_miles=stone_to_project_miles,
    concrete_to_project_miles=concrete_to_project_miles,
    crushed_concrete_to_reuse_miles=crushed_concrete_to_reuse_miles,
    rebar_to_project_miles=rebar_to_project_miles,
    nucor_to_hymmco_miles=nucor_to_hymmco_miles,
    include_virgin_aggregate_credit=False,
)


## Run Module 4

This uses the in-memory feasible results from Modules 1 and 2. If those cells have not been run, run Modules 1 and 2 first.


In [20]:
module_4_lca_results = run_module_4_lca(
    module_1_results=feasible_results,
    module_2_results=module_2_feasible_results,
    unit_impacts=lca_unit_impacts,
    inputs=lca_inputs,
)

module_4_lca_df = pd.DataFrame(module_4_lca_results)

print(f'LCA alternatives evaluated: {len(module_4_lca_results):,}')


LCA alternatives evaluated: 540


## Save Module 4 outputs


In [21]:
write_lca_results_csv(module_4_lca_results, output_dir / 'module_4_lca_results.csv')

print('Saved:')
print(output_dir / 'module_4_lca_results.csv')


Saved:
outputs\module_4_lca_results.csv


## Review lowest GWP alternatives


In [22]:
lca_review_columns = [
    'module',
    'subbase_thickness_in',
    'pavement_thickness_in',
    'bar_size',
    'bar_spacing_in',
    'fe3_psi',
    'gwp_concrete_kgco2e',
    'gwp_57_stone_kgco2e',
    'gwp_reinforcing_steel_kgco2e',
    'gwp_tufstrand_sf_kgco2e',
    'gwp_transport_kgco2e',
    'haul_rebar_ton_miles',
    'haul_rebar_nucor_to_hymmco_ton_miles',
    'haul_rebar_hymmco_to_project_ton_miles',
    'gwp_demolition_kgco2e',
    'gwp_crushing_kgco2e',
    'gwp_total_project_kgco2e',
]

available_lca_columns = [c for c in lca_review_columns if c in module_4_lca_df.columns]

module_4_lca_df[available_lca_columns].sort_values(
    by=['gwp_total_project_kgco2e']
).head(20)


,module,subbase_thickness_in,pavement_thickness_in,bar_size,bar_spacing_in,fe3_psi,gwp_concrete_kgco2e,gwp_57_stone_kgco2e,gwp_reinforcing_steel_kgco2e,gwp_tufstrand_sf_kgco2e,gwp_transport_kgco2e,haul_rebar_ton_miles,haul_rebar_nucor_to_hymmco_ton_miles,haul_rebar_hymmco_to_project_ton_miles,gwp_demolition_kgco2e,gwp_crushing_kgco2e,gwp_total_project_kgco2e
400,FRC,4,6,NaN,NaN,100.0,565370.37037,7166.666667,0.0,6674.86372,16035.543804,0.0,0.0,0.0,4777.777778,4837.5,604862.722340
401,FRC,4,6,NaN,NaN,150.0,565370.37037,7166.666667,0.0,7564.84555,16035.543804,0.0,0.0,0.0,4777.777778,4837.5,605752.704169
402,FRC,4,6,NaN,NaN,200.0,565370.37037,7166.666667,0.0,10902.27741,16035.543804,0.0,0.0,0.0,4777.777778,4837.5,609090.136029
428,FRC,6,6,NaN,NaN,100.0,565370.37037,10750.000000,0.0,6674.86372,18445.821667,0.0,0.0,0.0,4777.777778,4837.5,610856.333536
429,FRC,6,6,NaN,NaN,150.0,565370.37037,10750.000000,0.0,7564.84555,18445.821667,0.0,0.0,0.0,4777.777778,4837.5,611746.315365
403,FRC,4,6,NaN,NaN,250.0,565370.37037,7166.666667,0.0,14239.70927,16035.543804,0.0,0.0,0.0,4777.777778,4837.5,612427.567889
430,FRC,6,6,NaN,NaN,200.0,565370.37037,10750.000000,0.0,10902.27741,18445.821667,0.0,0.0,0.0,4777.777778,4837.5,615083.747225
404,FRC,4,6,NaN,NaN,300.0,565370.37037,7166.666667,0.0,17577.14113,16035.543804,0.0,0.0,0.0,4777.777778,4837.5,615764.999749
456,FRC,8,6,NaN,NaN,100.0,565370.37037,14333.333333,0.0,6674.86372,20856.099530,0.0,0.0,0.0,4777.777778,4837.5,616849.944732
457,FRC,8,6,NaN,NaN,150.0,565370.37037,14333.333333,0.0,7564.84555,20856.099530,0.0,0.0,0.0,4777.777778,4837.5,617739.926561


## Module 5 - Uncertainty and Sensitivity Analysis

Module 5 runs Monte Carlo uncertainty analysis and Spearman rank sensitivity analysis for all feasible alternatives. It evaluates uncertainty in total present worth and total project GWP.


In [23]:
from module_5 import (
    UncertaintyInputs,
    build_default_uncertain_parameters,
    run_module_5_uncertainty,
)
from module_5.uncertainty_design import write_results_csv as write_uncertainty_results_csv


## Module 5 setup

The default run uses 50,000 simulations and random seed 42. For quick debugging, temporarily reduce `n_simulations`, then set it back to 50,000 for the final run.


In [24]:
uncertainty_inputs = UncertaintyInputs(
    n_simulations=50_000,
    random_seed=42,
    end_of_service_life_yr=lcc_inputs.end_of_service_life_yr,
    concrete_ton_per_cy=lcc_inputs.concrete_ton_per_cy,
    stone_ton_per_cy=lca_inputs.stone_ton_per_cy,
)

uncertain_parameters = build_default_uncertain_parameters()

pd.DataFrame([p.__dict__ for p in uncertain_parameters])


,name,distribution,low,high,mean,std,alpha,beta,units,reference,description
0,concrete_unit_cost,uniform,150.000,230.00,NaN,NaN,NaN,NaN,$/cy,"RSMeans line item, user-selected edition; rang...",Ready-mix concrete unit cost.
1,stone_57_unit_cost,uniform,45.000,75.00,NaN,NaN,NaN,NaN,$/cy,"RSMeans line item, user-selected edition; rang...",#57 stone unit cost.
2,rebar_unit_cost,uniform,2600.000,3900.00,NaN,NaN,NaN,NaN,$/ton,"RSMeans line item, user-selected edition; rang...",Reinforcing steel unit cost.
3,fiber_unit_cost,uniform,1.323,1.47,NaN,NaN,NaN,NaN,$/lb,Parsons Corporation Euclid Chemical Pricing Ag...,TUF-STRAND SF unit cost with volume-discount u...
4,demolition_cost,uniform,32.000,55.00,NaN,NaN,NaN,NaN,$/cy,"RSMeans demolition line item, user-selected ed...",End-of-life concrete demolition cost.
5,crushing_cost,uniform,6.000,13.00,NaN,NaN,NaN,NaN,$/ton,RSMeans crushing/recycling line item or local ...,End-of-life concrete crushing cost.
6,recycled_aggregate_credit,uniform,3.000,9.00,NaN,NaN,NaN,NaN,$/ton,Local recycled aggregate value; range for unce...,Credit for demolished concrete reused as crush...
7,discount_rate,normal_trunc,0.000,0.08,0.03,0.01,NaN,NaN,decimal,FHWA LCCA practice; bounded real discount-rate...,Real discount rate used for end-of-life presen...
8,concrete_gwp_factor,beta_scaled,0.800,1.25,NaN,NaN,2.0,3.0,multiplier,ecoinvent/APOS concrete process factor uncerta...,Multiplier on concrete production GWP factor.
9,stone_57_gwp_factor,beta_scaled,0.750,1.35,NaN,NaN,2.0,3.0,multiplier,ecoinvent/APOS crushed aggregate process facto...,Multiplier on #57 stone production GWP factor.


## Run Module 5

This cell uses the LCC and LCA results from Modules 3 and 4. Run Modules 1 through 4 first.


In [25]:
(
    module_5_parameter_table,
    module_5_uncertainty_summary,
    module_5_spearman_sensitivity,
) = run_module_5_uncertainty(
    lcc_results=module_3_lcc_results,
    lca_results=module_4_lca_results,
    parameters=uncertain_parameters,
    inputs=uncertainty_inputs,
)

module_5_parameter_df = pd.DataFrame(module_5_parameter_table)
module_5_summary_df = pd.DataFrame(module_5_uncertainty_summary)
module_5_sensitivity_df = pd.DataFrame(module_5_spearman_sensitivity)

print(f'Uncertainty alternatives evaluated: {len(module_5_summary_df):,}')
print(f'Sensitivity rows: {len(module_5_sensitivity_df):,}')


Uncertainty alternatives evaluated: 540
Sensitivity rows: 15,120


## Save Module 5 outputs


In [26]:
write_uncertainty_results_csv(
    module_5_parameter_table,
    output_dir / 'module_5_uncertainty_parameter_table.csv',
)

write_uncertainty_results_csv(
    module_5_uncertainty_summary,
    output_dir / 'module_5_uncertainty_summary.csv',
)

write_uncertainty_results_csv(
    module_5_spearman_sensitivity,
    output_dir / 'module_5_spearman_sensitivity.csv',
)

print('Saved Module 5 outputs to the outputs folder.')


Saved Module 5 outputs to the outputs folder.


## Review Module 5 uncertainty summary


In [27]:
module_5_summary_df.sort_values(
    by=['total_present_worth_mean']
).head(20)


,alternative_id,module,subbase_thickness_in,pavement_thickness_in,bar_size,bar_spacing_in,fe3_psi,n_simulations,random_seed,total_present_worth_mean,...,total_present_worth_p95,total_present_worth_min,total_present_worth_max,gwp_total_project_kgco2e_mean,gwp_total_project_kgco2e_std,gwp_total_project_kgco2e_p05,gwp_total_project_kgco2e_p50,gwp_total_project_kgco2e_p95,gwp_total_project_kgco2e_min,gwp_total_project_kgco2e_max
400,401,FRC,4,6,NaN,NaN,100.0,50000,42,393870.456002,...,456054.931100,302529.788441,531358.921123,593608.145156,50871.562003,516516.236736,590042.228036,683206.927804,489485.441789,740835.690979
401,402,FRC,4,6,NaN,NaN,150.0,50000,42,394759.999584,...,456932.311135,303386.198545,532289.631672,594498.126985,50871.562003,517406.218566,590932.209866,684096.909633,490375.423619,741725.672809
402,403,FRC,4,6,NaN,NaN,200.0,50000,42,398095.788020,...,460290.057348,306597.736433,535779.796229,597835.558845,50871.562003,520743.650426,594269.641726,687434.341493,493712.855479,745063.104669
403,404,FRC,4,6,NaN,NaN,250.0,50000,42,401431.576455,...,463648.873818,309809.274322,539269.960787,601172.990705,50871.562003,524081.082286,597607.073586,690771.773354,497050.287339,748400.536529
404,405,FRC,4,6,NaN,NaN,300.0,50000,42,404767.364891,...,467000.315752,313020.812210,542760.125345,604510.422565,50871.562003,527418.514146,600944.505446,694109.205214,500387.719199,751737.968389
405,406,FRC,4,6,NaN,NaN,350.0,50000,42,408103.153327,...,470319.217061,316232.350099,546250.289903,607847.854426,50871.562003,530755.946006,604281.937306,697446.637074,503725.151059,755075.400249
406,407,FRC,4,6,NaN,NaN,400.0,50000,42,411438.941762,...,473676.672594,319443.887987,549740.454461,611185.286286,50871.562003,534093.377867,607619.369167,700784.068934,507062.582919,758412.832110
428,429,FRC,6,6,NaN,NaN,100.0,50000,42,425728.021562,...,491125.073829,327315.667407,569246.319865,599564.992029,50889.337439,522462.871130,596041.774145,689150.274693,494638.318099,747018.639880
429,430,FRC,6,6,NaN,NaN,150.0,50000,42,426617.565145,...,492015.420438,328172.077510,570177.030414,600454.973859,50889.337439,523352.852959,596931.755974,690040.256522,495528.299928,747908.621709
430,431,FRC,6,6,NaN,NaN,200.0,50000,42,429953.353580,...,495355.686697,331383.615399,573667.194972,603792.405719,50889.337439,526690.284819,600269.187834,693377.688383,498865.731789,751246.053569


## Review strongest sensitivity results


In [28]:
module_5_sensitivity_df.sort_values(
    by=['abs_spearman_rho'],
    ascending=False,
).head(30)


,alternative_id,output,parameter,spearman_rho,abs_spearman_rho
11810,422,gwp_total_project_kgco2e,concrete_gwp_factor,0.999383,0.999383
11838,423,gwp_total_project_kgco2e,concrete_gwp_factor,0.999383,0.999383
11866,424,gwp_total_project_kgco2e,concrete_gwp_factor,0.999383,0.999383
11894,425,gwp_total_project_kgco2e,concrete_gwp_factor,0.999383,0.999383
11922,426,gwp_total_project_kgco2e,concrete_gwp_factor,0.999383,0.999383
11950,427,gwp_total_project_kgco2e,concrete_gwp_factor,0.999383,0.999383
11978,428,gwp_total_project_kgco2e,concrete_gwp_factor,0.999383,0.999383
11782,421,gwp_total_project_kgco2e,concrete_gwp_factor,0.999342,0.999342
11670,417,gwp_total_project_kgco2e,concrete_gwp_factor,0.999342,0.999342
11614,415,gwp_total_project_kgco2e,concrete_gwp_factor,0.999342,0.999342


## Module 6 - Optimization and Selection

Module 6 applies the final selection rule: lowest LCA controls as long as the alternative is within 120% of the lowest deterministic LCC solution. Module 5 uncertainty and sensitivity results are attached only for the reported solutions.


In [29]:
from module_6 import SelectionInputs, run_module_6_selection
from module_6.selection_design import write_results_csv as write_selection_results_csv


## Run Module 6

This cell uses deterministic Module 3 LCC and Module 4 LCA for selection. Module 5 outputs are used only to report uncertainty and sensitivity for the selected solutions.


In [30]:
selection_inputs = SelectionInputs(
    cost_threshold_multiplier=1.20,
)

(
    module_6_benchmark_solution,
    module_6_selected_solutions,
    module_6_eligible_solutions,
    module_6_selected_sensitivity,
) = run_module_6_selection(
    lcc_results=module_3_lcc_results,
    lca_results=module_4_lca_results,
    uncertainty_summary=module_5_uncertainty_summary,
    sensitivity_results=module_5_spearman_sensitivity,
    inputs=selection_inputs,
)

module_6_benchmark_df = pd.DataFrame(module_6_benchmark_solution)
module_6_selected_df = pd.DataFrame(module_6_selected_solutions)
module_6_eligible_df = pd.DataFrame(module_6_eligible_solutions)
module_6_selected_sensitivity_df = pd.DataFrame(module_6_selected_sensitivity)

print(f'Eligible alternatives within 120% LCC threshold: {len(module_6_eligible_df):,}')
print(f'Reported selected/benchmark alternatives: {len(module_6_selected_df):,}')


Eligible alternatives within 120% LCC threshold: 18
Reported selected/benchmark alternatives: 2


## Save Module 6 outputs


In [31]:
write_selection_results_csv(
    module_6_benchmark_solution,
    output_dir / 'module_6_benchmark_lowest_lcc_solution.csv',
)

write_selection_results_csv(
    module_6_selected_solutions,
    output_dir / 'module_6_selected_solutions.csv',
)

write_selection_results_csv(
    module_6_eligible_solutions,
    output_dir / 'module_6_eligible_solutions.csv',
)

write_selection_results_csv(
    module_6_selected_sensitivity,
    output_dir / 'module_6_selected_solution_sensitivity.csv',
)

print('Saved Module 6 outputs to the outputs folder.')


Saved Module 6 outputs to the outputs folder.


## Review selected solutions


In [32]:
selection_review_columns = [
    'selection_role',
    'selection_status',
    'module',
    'subbase_thickness_in',
    'pavement_thickness_in',
    'bar_size',
    'bar_spacing_in',
    'fe3_psi',
    'total_present_worth',
    'lcc_percent_of_benchmark',
    'gwp_total_project_kgco2e',
    'total_present_worth_p05',
    'total_present_worth_p50',
    'total_present_worth_p95',
    'gwp_total_project_kgco2e_p05',
    'gwp_total_project_kgco2e_p50',
    'gwp_total_project_kgco2e_p95',
]

available_selection_columns = [c for c in selection_review_columns if c in module_6_selected_df.columns]
module_6_selected_df[available_selection_columns]


,selection_role,selection_status,module,subbase_thickness_in,pavement_thickness_in,bar_size,bar_spacing_in,fe3_psi,total_present_worth,lcc_percent_of_benchmark,gwp_total_project_kgco2e,total_present_worth_p05,total_present_worth_p50,total_present_worth_p95,gwp_total_project_kgco2e_p05,gwp_total_project_kgco2e_p50,gwp_total_project_kgco2e_p95
0,Benchmark - lowest deterministic LCC,Within 120% LCC threshold,FRC,4,6,NaN,NaN,100.0,385057.354735,100.000000,604862.72234,331374.896174,393963.320909,456054.931100,516516.236736,590042.228036,683206.927804
1,Override SRC - best LCA outside 120% LCC,Outside 120% LCC threshold; shown for SRC comp...,SRC,4,6,#4,8.0,NaN,605718.600654,157.306072,656400.17349,539523.495505,618272.392553,696910.150246,566520.111028,641094.005000,735059.703250


## Review selected-solution sensitivities


In [33]:
module_6_selected_sensitivity_df.sort_values(
    by=['alternative_id', 'output', 'sensitivity_rank']
).groupby(['alternative_id', 'output']).head(5)


,alternative_id,output,parameter,spearman_rho,abs_spearman_rho,sensitivity_rank
0,2,gwp_total_project_kgco2e,concrete_gwp_factor,0.990571,0.990571,1
1,2,gwp_total_project_kgco2e,rebar_gwp_factor,0.126375,0.126375,2
2,2,gwp_total_project_kgco2e,trucking_gwp_factor,0.036059,0.036059,3
3,2,gwp_total_project_kgco2e,crushing_gwp_factor,0.020900,0.020900,4
4,2,gwp_total_project_kgco2e,demolition_gwp_factor,0.017035,0.017035,5
14,2,total_present_worth,concrete_unit_cost,0.775720,0.775720,1
15,2,total_present_worth,rebar_unit_cost,0.548310,0.548310,2
16,2,total_present_worth,discount_rate,-0.191639,0.191639,3
17,2,total_present_worth,stone_57_unit_cost,0.179998,0.179998,4
18,2,total_present_worth,demolition_cost,0.045399,0.045399,5


## Module 7 - Summary Output Graphics

Module 7 is reserved for future summary graphics generated from Modules 1 through 6. Current status: **Not Yet Implemented**.

Planned outputs may include:

- Structural feasible/non-feasible alternatives by thickness
- LCC vs LCA tradeoff scatter plots
- Present-worth and GWP bar charts for selected alternatives
- Monte Carlo uncertainty interval plots
- Spearman sensitivity ranking graphics
- Final selected-alternative summary figure


In [34]:
from module_7 import SummaryOutputInputs, run_module_7_summary_output

summary_output_inputs = SummaryOutputInputs(output_dir=output_dir)

print('Module 7: Summary Output Graphics - Not Yet Implemented')

Module 7: Summary Output Graphics - Not Yet Implemented
